## 1. Read Gold Taxi Daily Fact

Read the final Gold Taxi Daily Fact table to determine the date range for the Date Dimension.

In [ ]:
df_fact = spark.table("dbo.gold_fact_taxi_daily_ready")

## 2. Determine Date Range

Calculate the minimum and maximum trip dates from the Gold Taxi Daily Fact table.

These dates define the calendar range for the Date Dimension.

In [ ]:
from pyspark.sql import functions as F

df_range = df_fact.agg(
    F.min("trip_date").alias("start_date"),
    F.max("trip_date").alias("end_date")
)

df_range.show()

## 3. Generate Date Sequence

Generate a continuous sequence of calendar dates between the minimum and maximum trip dates using the Spark sequence() function.

In [ ]:
df_dates_array = df_range.selectExpr(
    "sequence(start_date, end_date, interval 1 day) AS dates"
)

df_dates_array.show(truncate=False)

## 4. Expand Date Array into Rows

Convert the generated date array into individual rows using Spark explode(). Each row represents one calendar date.

In [ ]:
df_dates = df_dates_array.selectExpr(
    "explode(dates) AS date"
)

df_dates.show(5)
print("Total rows:", df_dates.count())

## 5. Generate DateKey

Create a unique integer date key in yyyyMMdd format for each calendar date.

In [ ]:

df_dim_date = df_dates.withColumn(
    "DateKey",
    F.date_format("date", "yyyyMMdd").cast("int")
)

df_dim_date.show(5)

## 6. Add Calendar Attributes

Extract year, month, and quarter from each date to support time-based analysis and reporting.

In [ ]:
df_dim_date = (
    df_dim_date
    .withColumn("Year", F.year("date"))
    .withColumn("Month", F.month("date"))
    .withColumn("Quarter", F.quarter("date"))
)

df_dim_date.show(5)

## 7. Add Weekday and Weekend Attributes

Calculate the day of the week (Monday = 1, Sunday = 7) and identify weekends for weekday-versus-weekend analysis.

In [ ]:
df_dim_date = (
    df_dim_date
    .withColumn("DayOfWeek", F.weekday("date") + 1)
    .withColumn("IsWeekend", F.col("DayOfWeek").isin(6, 7))
)

df_dim_date.show(10)

## 8. Add Month Name

Extract the full month name to support readable labels in analytical reports and dashboards.

In [ ]:
df_dim_date = df_dim_date.withColumn(
    "MonthName",
    F.date_format("date", "MMMM")
)

df_dim_date.show(10)

## 9. Add Day Name

Extract the full weekday name to support daily and weekend analysis in reporting.

In [ ]:
df_dim_date = df_dim_date.withColumn(
    "DayName",
    F.date_format("date", "EEEE")
)

df_dim_date.select(
    "date", "DayOfWeek", "DayName", "IsWeekend"
).show(7)

In [ ]:
print("Total rows:", df_dim_date.count())

print(
    "Unique DateKeys:",
    df_dim_date.select("DateKey").distinct().count()
)

## 10. Validate Date Dimension

Perform data quality checks to ensure:
- DateKey values are unique and not NULL.
- Calendar attributes contain no NULL values.

In [ ]:
df_null_keys = df_dim_date.filter(
    F.col("DateKey").isNull()
)

print("NULL DateKeys:", df_null_keys.count())

In [ ]:
df_dim_date.selectExpr(
    "SUM(CASE WHEN Year IS NULL Then 1 else 0 end) as null_year"
).show()

df_dim_date.selectExpr(
    "SUM(CASE WHEN Month IS NULL Then 1 else 0 end) as null_month"
).show()

df_dim_date.selectExpr(
    "SUM(CASE WHEN Quarter IS NULL Then 1 else 0 end) as null_quarter"
).show()

## 11. Save and Verify Gold Date Dimension

Save the validated Date Dimension as a Delta table in the Gold layer and verify the persisted data.

In [ ]:
df_dim_date.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("dbo.gold_dim_date")

In [ ]:
df_saved = spark.table("dbo.gold_dim_date")

df_saved.show(5)
print("Saved rows:", df_saved.count())